<a href="https://colab.research.google.com/github/DimDragg/-/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0_%D1%80%D0%BE%D0%B1%D0%BE%D1%82%D0%B0_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Рекомендаційні алгоритми, Чуркін Дмитро ФІТ 3-15

Був присутній на парі

Завдання: доробити код з kaggle (LR_7_Rec_Sist), також середовище виконання GPU 4. Це вже моя ініціатива, щоб гарно було: замість однієї главної одиниці вимірювання замінити на варіант у списку (15)

In [2]:
!pip install numpy==1.26.4
!pip install scikit-surprise

In [3]:
from surprise import Dataset, Reader
from surprise import SVD, KNNBasic
from surprise.model_selection import GridSearchCV, train_test_split
from surprise import accuracy

import pandas as pd
import numpy as np
import sys
import io

# Завантаження датасету
!wget --no-check-certificate https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -o ml-100k.zip

# Читання даних
data = pd.read_csv(
    'ml-100k/u.data',
    sep='\t',
    names=['user_id', 'movie_id', 'rating', 'timestamp']
)

print(data.head())

# Масштабування рейтингу (1–15)
data['rating'] = data['rating'] * 3  # 5 → 15

# Surprise Dataset
reader = Reader(rating_scale=(1, 15))
data_surprise = Dataset.load_from_df(
    data[['user_id', 'movie_id', 'rating']],
    reader
)

trainset, testset = train_test_split(data_surprise, test_size=0.2)

# GridSearch SVD
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.05]
}

grid_search_svd = GridSearchCV(SVD, param_grid_svd, measures=['rmse', 'mae'], cv=3)
grid_search_svd.fit(data_surprise)

print("Best RMSE SVD:", grid_search_svd.best_score['rmse'])
print("Best params SVD:", grid_search_svd.best_params['rmse'])

# GridSearch KNN
param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [True, False]
    }
}

backup = sys.stdout
sys.stdout = io.StringIO()

grid_search_knn = GridSearchCV(KNNBasic, param_grid_knn, measures=['rmse', 'mae'], cv=3)
grid_search_knn.fit(data_surprise)

sys.stdout = backup

print("Best RMSE KNN:", grid_search_knn.best_score['rmse'])
print("Best params KNN:", grid_search_knn.best_params['rmse'])

# Навчання найкращих моделей
best_svd = SVD(**grid_search_svd.best_params['rmse'])
best_svd.fit(trainset)

pred_svd = best_svd.test(testset)
print("SVD RMSE:", accuracy.rmse(pred_svd))

best_knn = KNNBasic(**grid_search_knn.best_params['rmse'])
best_knn.fit(trainset)

pred_knn = best_knn.test(testset)
print("KNN RMSE:", accuracy.rmse(pred_knn))

# Вибір найкращої моделі
if grid_search_svd.best_score['mae'] < grid_search_knn.best_score['mae']:
    best_model = best_svd
    print("Краща модель: SVD")
else:
    best_model = best_knn
    print("Краща модель: KNN")

# Рекомендації
best_model.fit(trainset)

user_id = data['user_id'].iloc[0]

inner_uid = trainset.to_inner_uid(user_id)
user_items = trainset.ur[inner_uid]

all_items = set(trainset.all_items())
rated_items = set([i for (i, _) in user_items])
unrated_items = all_items - rated_items

predictions = [
    (item, best_model.predict(user_id, trainset.to_raw_iid(item)).est)
    for item in unrated_items
]

predictions.sort(key=lambda x: x[1], reverse=True)

print("\nТОП-10 рекомендацій:")
for item, score in predictions[:10]:
    print(f"Фільм {trainset.to_raw_iid(item)} | рейтинг: {score:.2f}")

--2026-04-14 15:26:25--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip.1’

ml-100k.zip.1       100%[===================>]   4.70M  9.18MB/s    in 0.5s    

2026-04-14 15:26:26 (9.18 MB/s) - ‘ml-100k.zip.1’ saved [4924029/4924029]

Archive:  ml-100k.zip
  inflating: ml-100k/allbut.pl       
  inflating: ml-100k/mku.sh          
  inflating: ml-100k/README          
  inflating: ml-100k/u.data          
  inflating: ml-100k/u.genre         
  inflating: ml-100k/u.info          
  inflating: ml-100k/u.item          
  inflating: ml-100k/u.occupation    
  inflating: ml-100k/u.user          
  inflating: ml-100k/u1.base         
  inflating: ml-100k/u1.test         
  inflating: ml-100k/u2.base         
  infl

У ході лабораторної роботи було реалізовано систему рекомендацій на основі датасету MovieLens. Дані були попередньо оброблені та масштабовані (шкала оцінок змінена з 5 до 15), після чого використано бібліотеку Surprise для побудови моделей.

Було досліджено два алгоритми рекомендацій:

1)SVD (матрична факторизація);

2)KNNBasic (метод найближчих сусідів).

Для кожної моделі проведено підбір гіперпараметрів за допомогою GridSearchCV та оцінено якість за метриками RMSE та MAE. У результаті встановлено, що одна з моделей (залежно від отриманих значень MAE) демонструє кращу точність прогнозування.

Також було реалізовано механізм генерації рекомендацій: для обраного користувача визначено фільми, які він ще не оцінював, і побудовано список з 10 найкращих рекомендацій на основі прогнозованих рейтингів.